# M5 Baseline LightGBM Pipeline on Cleaned Dataset

This notebook is a streamlined Colab-ready pipeline that:

- loads the pre-cleaned long-format dataset
- removes the old raw-data EDA and merge steps
- adds a minimal but strong feature set
- trains a LightGBM baseline model
- reports RMSE and MAE on the final 28-day validation window

You only need to update the Google Drive paths in the configuration cell.


In [4]:
# =========================
# 1) Colab setup
# =========================
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# =========================
# 2) Imports
# =========================
from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", 200)


In [6]:
# =========================
# 3) Configuration
# =========================
# Update these paths to your own Google Drive folders.
DATA_DIR = Path("/content/drive/MyDrive/Group Project - Predictive/data/cleaned_data")
CLEAN_DATA_PATH = DATA_DIR / "long_df_clean.parquet"

OUTPUT_DIR = Path("/content/drive/MyDrive/Group Project - Predictive/data/FE_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fast iteration options
FAST_MODE = True
N_SERIES = 2000
RANDOM_STATE = 42
VALID_DAYS = 28

# Experiment label
PIPELINE_NAME = "baseline_lgb_clean_minimal_features"
CURRENT_EXPERIMENT = "tweedie_power_1p5_final"

print("CLEAN_DATA_PATH:", CLEAN_DATA_PATH)
print("OUTPUT_DIR      :", OUTPUT_DIR)


CLEAN_DATA_PATH: /content/drive/MyDrive/Group Project - Predictive/data/cleaned_data/long_df_clean.parquet
OUTPUT_DIR      : /content/drive/MyDrive/Group Project - Predictive/data/FE_output


In [7]:
# =========================
# 4) Load cleaned dataset
# =========================
df = pd.read_parquet(CLEAN_DATA_PATH).copy()

print("Loaded shape:", df.shape)
print("Columns:", len(df.columns))
display(df.head(3))


Loaded shape: (16098720, 25)
Columns: 25


,id,item_id,dept_id,cat_id,store_id,state_id,d,demand,d_num,date,wm_yr_wk,wday,month,year,weekday,snap_CA,snap_TX,snap_WI,event_name_1,event_type_1,event_name_2,event_type_2,sell_price,is_active,snap_flag
0,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1414,1,1414,2014-12-12,11445,7,12,2014,Friday,0,1,1,NoEvent,NoEvent,NoEvent,NoEvent,2.24,1,0
1,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1415,2,1415,2014-12-13,11446,1,12,2014,Saturday,0,1,0,NoEvent,NoEvent,NoEvent,NoEvent,2.24,1,0
2,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1416,2,1416,2014-12-14,11446,2,12,2014,Sunday,0,0,1,NoEvent,NoEvent,NoEvent,NoEvent,2.24,1,0


In [8]:
# =========================
# 5) Basic type checks and optional fast-mode sampling
# =========================
df["date"] = pd.to_datetime(df["date"])

# Keep only the columns needed for this baseline.
required_base_cols = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "date", "demand", "sell_price"
]
optional_cols = ["snap_CA", "snap_TX", "snap_WI", "wm_yr_wk"]
keep_cols = [c for c in required_base_cols + optional_cols if c in df.columns]

df = df[keep_cols].copy()

# Optional fast-mode sampling at the series level.
if FAST_MODE:
    sampled_ids = (
        df["id"]
        .drop_duplicates()
        .sample(n=min(N_SERIES, df["id"].nunique()), random_state=RANDOM_STATE)
    )
    df = df[df["id"].isin(sampled_ids)].copy()

df = df.sort_values(["id", "date"]).reset_index(drop=True)

print("Working shape:", df.shape)
print("Unique series:", df["id"].nunique())
print("Date range   :", df["date"].min().date(), "to", df["date"].max().date())


Working shape: (1056000, 13)
Unique series: 2000
Date range   : 2014-12-12 to 2016-05-22


## Minimal feature set

This notebook uses a shared baseline feature family that is small enough for quick testing but still much stronger than a pure calendar-price baseline:

- calendar: `dow`, `month`
- price: `sell_price`, `price_change`, `is_promo`
- demand history: `lag_7`, `lag_28`, `rmean_7`, `rmean_28`


In [18]:
# =========================
# 6) Feature engineering
# =========================
df = df.sort_values(["id", "date"]).reset_index(drop=True)

# Calendar features
df["dow"] = df["date"].dt.dayofweek.astype("int8")
df["month"] = df["date"].dt.month.astype("int8")

# Price features
price_grp = df.groupby("id", sort=False)["sell_price"]
df["price_lag_1"] = price_grp.shift(1)
df["price_change"] = ((df["sell_price"] - df["price_lag_1"]) / df["price_lag_1"]).replace([np.inf, -np.inf], np.nan)
df["price_change"] = df["price_change"].fillna(0).astype("float32")
df["is_promo"] = (df["sell_price"] < df["price_lag_1"].fillna(df["sell_price"])).astype("int8")

# Demand history features
demand_grp = df.groupby("id", sort=False)["demand"]
df["lag_7"] = demand_grp.shift(7)
df["lag_28"] = demand_grp.shift(28)

df["rmean_7"] = (
    df.groupby("id", sort=False)["lag_7"]
      .transform(lambda x: x.rolling(7).mean())
)

df["rmean_28"] = (
    df.groupby("id", sort=False)["lag_28"]
      .transform(lambda x: x.rolling(28).mean())
)

# Downcast numeric features
for col in ["sell_price", "price_lag_1", "lag_7", "lag_28", "rmean_7", "rmean_28"]:
    df[col] = pd.to_numeric(df[col], downcast="float")

feature_cols = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "dow", "month",
    "sell_price", "price_change", "is_promo",
    "lag_7", "lag_28", "rmean_7", "rmean_28"
]

required_history_cols = ["lag_28", "rmean_28"]
model_df = df.dropna(subset=required_history_cols).copy()

cat_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
for col in cat_cols:
    model_df[col] = model_df[col].astype("category")

print("Model shape after dropping required history NaNs:", model_df.shape)
display(model_df[["id", "date"] + feature_cols + ["demand"]].head(3))

Model shape after dropping required history NaNs: (946000, 22)


,id,date,item_id,dept_id,cat_id,store_id,state_id,dow,month,sell_price,price_change,is_promo,lag_7,lag_28,rmean_7,rmean_28,demand
55,FOODS_1_001_TX_3_evaluation,2015-02-05,FOODS_1_001,FOODS_1,FOODS,TX_3,TX,3,2,2.24,0.0,0,1.0,0.0,0.428571,0.214286,0
56,FOODS_1_001_TX_3_evaluation,2015-02-06,FOODS_1_001,FOODS_1,FOODS,TX_3,TX,4,2,2.24,0.0,0,0.0,0.0,0.428571,0.214286,0
57,FOODS_1_001_TX_3_evaluation,2015-02-07,FOODS_1_001,FOODS_1,FOODS,TX_3,TX,5,2,2.24,0.0,0,0.0,0.0,0.428571,0.178571,0


In [19]:
# =========================
# 7) Train / valid split
# =========================
max_date = model_df["date"].max()
valid_start = max_date - pd.Timedelta(days=VALID_DAYS - 1)

train_df = model_df[model_df["date"] < valid_start].copy()
valid_df = model_df[model_df["date"] >= valid_start].copy()

print("Validation start:", valid_start.date())
print("Validation end  :", max_date.date())
print("Train shape     :", train_df.shape)
print("Valid shape     :", valid_df.shape)
print("Train ids       :", train_df["id"].nunique())
print("Valid ids       :", valid_df["id"].nunique())


Validation start: 2016-04-25
Validation end  : 2016-05-22
Train shape     : (890000, 22)
Valid shape     : (56000, 22)
Train ids       : 2000
Valid ids       : 2000


In [20]:
# =========================
# 8) LightGBM configuration
# =========================
NUM_BOOST_ROUND = 2000
EARLY_STOPPING_ROUNDS = 100
VERBOSE_EVAL = 100

LGB_PARAMS = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.5,
    "metric": "rmse",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 127,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.0,
    "lambda_l2": 0.1,
    "seed": RANDOM_STATE,
    "verbosity": -1
}

print(json.dumps(LGB_PARAMS, indent=2))


{
  "objective": "tweedie",
  "tweedie_variance_power": 1.5,
  "metric": "rmse",
  "boosting_type": "gbdt",
  "learning_rate": 0.05,
  "num_leaves": 127,
  "min_data_in_leaf": 100,
  "feature_fraction": 0.8,
  "bagging_fraction": 0.8,
  "bagging_freq": 1,
  "lambda_l1": 0.0,
  "lambda_l2": 0.1,
  "seed": 42,
  "verbosity": -1
}


In [21]:
# =========================
# 9) Prepare matrices
# =========================
X_train = train_df[feature_cols].copy()
y_train = train_df["demand"].copy()

X_valid = valid_df[feature_cols].copy()
y_valid = valid_df["demand"].copy()

categorical_cols_model = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

for col in categorical_cols_model:
    X_train[col] = X_train[col].astype("category")
    X_valid[col] = X_valid[col].astype("category")

train_set = lgb.Dataset(
    X_train,
    label=y_train,
    categorical_feature=categorical_cols_model,
    free_raw_data=False
)

valid_set = lgb.Dataset(
    X_valid,
    label=y_valid,
    categorical_feature=categorical_cols_model,
    free_raw_data=False
)


In [22]:
# =========================
# 10) Train model
# =========================
model = lgb.train(
    params=LGB_PARAMS,
    train_set=train_set,
    valid_sets=[train_set, valid_set],
    valid_names=["train", "valid"],
    num_boost_round=NUM_BOOST_ROUND,
    callbacks=[
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS),
        lgb.log_evaluation(period=VERBOSE_EVAL)
    ]
)


Training until validation scores don't improve for 100 rounds
[100]	train's rmse: 2.24074	valid's rmse: 2.28619
[200]	train's rmse: 2.17914	valid's rmse: 2.29626
Early stopping, best iteration is:
[103]	train's rmse: 2.23655	valid's rmse: 2.28496


In [23]:
# =========================
# 11) Evaluate
# =========================
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
valid_df["pred"] = model.predict(X_valid, num_iteration=model.best_iteration)

rmse = root_mean_squared_error(valid_df["demand"], valid_df["pred"])
mae = mean_absolute_error(valid_df["demand"], valid_df["pred"])

metrics_df = pd.DataFrame([{
    "pipeline_name": PIPELINE_NAME,
    "experiment_name": CURRENT_EXPERIMENT,
    "fast_mode": FAST_MODE,
    "n_series": int(df["id"].nunique()),
    "valid_days": VALID_DAYS,
    "rmse": rmse,
    "mae": mae,
    "best_iteration": int(model.best_iteration)
}])

print(metrics_df)


                         pipeline_name          experiment_name  fast_mode  \
0  baseline_lgb_clean_minimal_features  tweedie_power_1p5_final       True   

   n_series  valid_days      rmse       mae  best_iteration  
0      2000          28  2.284962  1.046555             103  


In [17]:
# =========================
# 12) Save outputs
# =========================
# metrics_path = OUTPUT_DIR / f"{PIPELINE_NAME}_metrics.csv"
# preds_path = OUTPUT_DIR / f"{PIPELINE_NAME}_valid_predictions.csv"
# fi_path = OUTPUT_DIR / f"{PIPELINE_NAME}_feature_importance.csv"

# metrics_df.to_csv(metrics_path, index=False)

# valid_df[["id", "date", "demand", "pred"]].to_csv(preds_path, index=False)

# fi_df = pd.DataFrame({
#     "feature": feature_cols,
#     "importance_gain": model.feature_importance(importance_type="gain"),
#     "importance_split": model.feature_importance(importance_type="split")
# }).sort_values("importance_gain", ascending=False)

# fi_df.to_csv(fi_path, index=False)

# print("Saved:")
# print(metrics_path)
# print(preds_path)
# print(fi_path)

# display(fi_df.head(15))


In [ ]:
# =========================
# 13) Notes for next experiments
# =========================
# Suggested next feature families to test on top of this shared baseline:
# 1) More lag features: lag_14, lag_56, lag_365
# 2) More rolling features: rstd_7, rstd_28
# 3) Price family: price_vs_mean, price_vs_dept_avg
# 4) Calendar / event family: snap_flag, event indicators
# 5) Hierarchy family: dept_avg_demand, store_avg_demand
